# Auditing a model on `german`

A full isitfair audit of a logistic regression fitted to the public
`german` dataset: discrimination, calibration, decision curves,
Hosmer-Lemeshow, estimability, a recalibration ladder, mitigation and an
HTML report.

Before running, build the prediction table once from the repository root:

    python -I extra_examples/datasets.py german

`B = 50` bootstrap replicates keep the run short; use 2,000 for results you
intend to report.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "extra_examples" / "datasets.py").exists():
        HERE = _c / "extra_examples"
        break
sys.path.insert(0, str(HERE))
import datasets  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing  # noqa: E402

NAME = "german"
B = 50
SEED = 42
META = datasets.meta(NAME)
THRESHOLD = META["threshold"]
ATTRS = datasets.ATTRS[NAME]
print(f"isitfair {isitfair.__version__}")
print(f"{NAME}: outcome {META['outcome']}; threshold {THRESHOLD}; B = {B}")

isitfair 0.4.0
german: outcome class == 2 (bad credit risk); threshold 0.3; B = 50


## 1. Load the predictions

One row per test-set patient or person: the outcome, the model's predicted
probability, and the attributes to audit.

In [2]:
df = datasets.load(NAME)
print(f"n = {len(df)}, prevalence = {df['y_true'].mean():.3f}")
for a in ATTRS:
    print(f"{a}: {df[a].nunique()} levels, {int(df[a].isna().sum())} missing")

n = 300, prevalence = 0.300
sex: 2 levels, 0 missing
age_group: 4 levels, 0 missing


isitfair refuses missing subgroup labels rather than silently dropping those
rows. If any attribute has missing labels, this cell shows the error and then
recodes them to an explicit `"Unknown"` level.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    try:
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED)
    except ValueError as exc:
        print("ValueError:", exc)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

recoded to 'Unknown': none


## 2. Split off a calibration half for mitigation

The test half is split 50/50, stratified on the outcome. Mitigation
calibrators are fitted on one half and every audit runs on the other, so no
row is scored by a calibrator fitted on it.

In [4]:
i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                 stratify=df["y_true"])
cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
# mitigate() rejects audited levels that the calibration half never saw; drop
# such rows (for example diabetes130's single "Unknown/Invalid" gender).
unseen = np.zeros(len(eval_df), bool)
for a in ATTRS:
    unseen |= ~eval_df[a].isin(set(cal_df[a]))
if unseen.any():
    print(f"dropping {int(unseen.sum())} audited row(s) with levels absent from the calibration half")
    eval_df = eval_df[~unseen].reset_index(drop=True)
print(f"calibration n = {len(cal_df)}, audit n = {len(eval_df)}")

calibration n = 150, audit n = 150


## 3. Construct the audit

Each attribute is audited on its own, and the first two are also crossed
(passing `None` for a tuple key asks isitfair to build the intersection).

In [5]:
y = eval_df["y_true"].to_numpy(dtype=float)
p = eval_df["y_proba"].to_numpy(dtype=float)
marginals = {a: eval_df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None

audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                      n_bootstrap=B, random_state=SEED)
audit

FairnessAudit(n=150, events=45, threshold=0.3, n_bootstrap=50, random_state=42, attributes: sex (2), age_group (4), sex × age_group (8))

## 4. Discrimination

In [6]:
disc = audit.discrimination()
disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                              "auroc_ci_low", "auroc_ci_high"]]

,attribute,subgroup,n,n_events,auroc,auroc_ci_low,auroc_ci_high
0,sex,Overall,150.0,45.0,0.817778,0.752898,0.884603
1,sex,female,54.0,19.0,0.824060,NaN,NaN
2,sex,male,96.0,26.0,0.813187,0.727003,0.879828
7,age_group,Overall,150.0,45.0,0.817778,0.752898,0.884603
8,age_group,19-24,28.0,13.0,0.717949,NaN,NaN
9,age_group,25-34,54.0,18.0,0.765432,NaN,NaN
10,age_group,35-49,44.0,8.0,0.934028,NaN,NaN
11,age_group,50+,24.0,6.0,0.851852,NaN,NaN
16,sex × age_group,Overall,150.0,45.0,0.817778,0.752898,0.884603
17,sex × age_group,female / 19-24,18.0,8.0,0.775000,NaN,NaN


## 5. Calibration

In [7]:
cal = audit.calibration()
cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                            "calibration_slope", "brier", "ece", "ici"]]

,attribute,subgroup,n,calibration_intercept,calibration_slope,brier,ece,ici
0,sex,Overall,150.0,0.036053,1.009493,0.149072,0.067452,0.050585
1,sex,female,54.0,0.273452,1.113371,0.155868,0.105858,0.072607
2,sex,male,96.0,-0.110764,0.955411,0.145249,0.089756,0.047884
5,age_group,Overall,150.0,0.036053,1.009493,0.149072,0.067452,0.050585
6,age_group,19-24,28.0,0.468388,0.594465,0.219423,0.232063,0.086150
7,age_group,25-34,54.0,0.138344,0.881756,0.174623,0.153305,0.089469
8,age_group,35-49,44.0,-0.358517,1.793101,0.090357,0.106395,0.074685
9,age_group,50+,24.0,-0.238642,1.136379,0.117149,0.146506,0.062292
12,sex × age_group,Overall,150.0,0.036053,1.009493,0.149072,0.067452,0.050585
13,sex × age_group,female / 19-24,18.0,0.452914,0.928375,0.186415,0.221217,0.092677


In [8]:
curves = audit.calibration_curves()
roc = audit.roc_curves()
print("calibration curves:", curves.shape, "| ROC curves:", roc.shape)

calibration curves: (1700, 7) | ROC curves: (1700, 7)


## 6. Decision curve: gaps in net benefit between subgroups

In [9]:
dc = audit.decision_curve()
dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]]

,attribute,gap_value,gap_ci_low,gap_ci_high
60,sex,0.098860,NaN,NaN
161,age_group,0.287768,NaN,NaN
342,sex × age_group,0.366071,NaN,NaN


## 7. Estimability

Cells with too few events for a metric are flagged rather than reported.

In [10]:
est = audit.estimability()
print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

159 of 208 cell-metrics not reportable


## 8. Hosmer-Lemeshow

In [11]:
hl = audit.hosmer_lemeshow()
hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                          "hl_p_value", "n_bins_used"]]

,attribute,subgroup,n,hl_statistic,hl_df,hl_p_value,n_bins_used
0,sex,Overall,150.0,9.557216,10.0,0.480160,10.0
1,sex,female,54.0,6.165551,10.0,0.801168,10.0
2,sex,male,96.0,9.435049,10.0,0.491372,10.0
4,age_group,Overall,150.0,9.557216,10.0,0.480160,10.0
5,age_group,19-24,28.0,14.088742,10.0,0.168982,10.0
6,age_group,25-34,54.0,12.512175,10.0,0.252239,10.0
7,age_group,35-49,44.0,4.654233,10.0,0.913048,10.0
8,age_group,50+,24.0,5.721221,10.0,0.838116,10.0
10,sex × age_group,Overall,150.0,9.557216,10.0,0.480160,10.0
11,sex × age_group,female / 19-24,18.0,10.503800,10.0,0.397458,10.0


## 9. Intersection cells

In [12]:
audit.cell_diagnostics() if INTERSECTION else "single attribute; no intersection"

,attribute,subgroup,n,n_events,shrunk,marginal_rate,cell_rate,shrinkage_factor
0,sex × age_group,female / 19-24,18,8,True,0.351852,0.444444,0.483871
1,sex × age_group,female / 25-34,22,9,True,0.351852,0.409091,0.434084
2,sex × age_group,female / 35-49,7,1,True,0.351852,0.142857,0.706806
3,sex × age_group,female / 50+,7,1,True,0.351852,0.142857,0.706806
4,sex × age_group,male / 19-24,10,5,True,0.270833,0.500000,0.627907
5,sex × age_group,male / 25-34,32,9,True,0.270833,0.281250,0.345269
6,sex × age_group,male / 35-49,37,7,True,0.270833,0.189189,0.313225
7,sex × age_group,male / 50+,17,5,True,0.270833,0.294118,0.498155


## 10. Recalibration ladder on the attribute with the largest intercept gap

A level with no events at all cannot be recalibrated, and isitfair says so.
In that case the ladder moves on to the next attribute.

In [13]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]
ladder = None
for by in RANKED:
    try:
        ladder = audit.recalibration_ladder(by=by)
        print("ladder by", by)
        break
    except ValueError as exc:
        print(f"{by}: {exc}")
ladder

ladder by age_group


,rung,attribute,subgroup,n,n_events,calibration_intercept,intercept_ci_low,intercept_ci_high,calibration_slope,slope_ci_low,...,brier_ci_low,brier_ci_high,ece,ici,net_benefit,gap_metric,gap_value,event_floor,below_floor,not_estimable_final
0,none,age_group,Overall,150.0,45.0,0.036053,-0.337500,0.373120,1.009493,NaN,...,0.119556,0.189396,0.067452,0.050585,0.151429,NaN,NaN,25.0,False,True
1,none,age_group,19-24,28.0,13.0,0.468388,NaN,NaN,0.594465,NaN,...,NaN,NaN,0.232063,0.086150,0.209184,NaN,NaN,25.0,True,True
2,none,age_group,25-34,54.0,18.0,0.138344,NaN,NaN,0.881756,NaN,...,NaN,NaN,0.153305,0.089469,0.153439,NaN,NaN,25.0,True,True
3,none,age_group,35-49,44.0,8.0,-0.358517,NaN,NaN,1.793101,NaN,...,NaN,NaN,0.106395,0.074685,0.110390,NaN,NaN,25.0,True,True
4,none,age_group,50+,24.0,6.0,-0.238642,NaN,NaN,1.136379,NaN,...,NaN,NaN,0.146506,0.062292,0.154762,NaN,NaN,25.0,True,True
5,none,age_group,GAP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,max_intercept_gap,0.826905,NaN,NaN,NaN
6,common,age_group,Overall,150.0,45.0,0.035498,-0.356386,0.460978,0.103113,NaN,...,0.128662,0.207215,0.139340,0.068507,0.158095,NaN,NaN,25.0,False,True
7,common,age_group,19-24,28.0,13.0,0.528778,NaN,NaN,0.046784,NaN,...,NaN,NaN,0.188965,0.133286,0.224490,NaN,NaN,25.0,True,True
8,common,age_group,25-34,54.0,18.0,0.119113,NaN,NaN,0.039827,NaN,...,NaN,NaN,0.219836,0.106146,0.158730,NaN,NaN,25.0,True,True
9,common,age_group,35-49,44.0,8.0,-0.401786,NaN,NaN,1.020213,NaN,...,NaN,NaN,0.147784,0.078168,0.129870,NaN,NaN,25.0,True,True


## 11. Mitigation

Group-wise isotonic recalibration, then Wasserstein post-processing, both
fitted on the calibration half.

In [14]:
calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba"].to_numpy(float),
              {a: cal_df[a].to_numpy(object) for a in ATTRS})
mitigated_gr = audit.mitigate(method=GroupRecalibration(method="isotonic"), calibration_data=calib_data)
a0 = ATTRS[0]
keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
before, after = audit.calibration(), mitigated_gr[a0].calibration()
print("Before,", a0)
print(before[before.attribute == a0][keep].to_string(index=False))
print("\nAfter group isotonic recalibration,", a0)
print(after[after.attribute == a0][keep].to_string(index=False))

Before, sex
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.036053           1.009493 0.149072 0.067452
  female               0.273452           1.113371 0.155868 0.105858
    male              -0.110764           0.955411 0.145249 0.089756
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN

After group isotonic recalibration, sex
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.081145           0.107554 0.153392 0.093455
  female               0.206955           0.099444 0.162903 0.125271
    male               0.013391           0.142407 0.148043 0.105469
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN


In [15]:
mitigated_w = audit.mitigate(method=WassersteinPostprocessing(random_state=SEED), calibration_data=calib_data)
d_after = mitigated_w[a0].discrimination()
d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
    ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]]

,gap_metric,gap_value,gap_ci_low,gap_ci_high
3,equal_opportunity_difference,0.085020,NaN,NaN
4,predictive_equality,0.000000,NaN,NaN
5,equalized_odds_gap,0.085020,NaN,NaN
6,statistical_parity,0.013889,NaN,NaN


## 12. HTML report

In [16]:
REPORT = HERE / "reports" / f"{NAME}.html"
REPORT.parent.mkdir(exist_ok=True)
audit.report(out=REPORT, title=f"{NAME}: isitfair audit",
             metadata={"Dataset": NAME, "Model": "LogisticRegression",
                       "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
             mitigation_audits=mitigated_gr)
print("wrote", REPORT.relative_to(HERE.parent))

wrote extra_examples\reports\german.html
